# Board defect detector (second training target)

Finds repair-relevant defects on circuit-board photos: corrosion, burn marks, bulging capacitors, cracked solder joints, worn rubber pads.

**Status:** planned. Until this model exists, the app finds defects with Claude vision (see `lib/defects.ts`). When `public/models/defects.onnx` exists, the app uses this model first and asks Claude only to explain. The UI doesn't change.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

Data plan:
1. Photos of broken boards from the repair flow (the app can export them from the debug panel), plus our own broken remotes and controllers.
2. Public PCB-defect datasets from Roboflow Universe. Most are factory soldering defects; keep only the classes below.
3. Label in Roboflow and export as YOLOv8.


In [ ]:
!pip -q install ultralytics roboflow
import shutil, yaml, re
from pathlib import Path
from ultralytics import YOLO


## 1. Class list

Must match `shared/defect_classes.json` exactly (checked by `scripts/check-classes.mjs`).

In [ ]:
# Keep in sync with shared/defect_classes.json
DEFECT_CLASSES = ["corroded", "burnt", "bulging_cap", "cracked_joint", "worn_pad"]

RULES = [
    (["corros", "rust", "oxid", "green"], "corroded"),
    (["burn", "scorch", "char"], "burnt"),
    (["bulg", "swollen", "cap"], "bulging_cap"),
    (["crack", "cold", "dry", "joint", "insufficient"], "cracked_joint"),
    (["pad", "rubber", "carbon"], "worn_pad"),
]
def to_canonical(name):
    low = re.sub(r"[^a-z]", "", name.lower())
    for keys, target in RULES:
        if any(k in low for k in keys):
            return target
    return None


## 2. Download labeled data (YOLOv8 format)

Add `(workspace, project, version)` entries the same way as in the parts notebook.

In [ ]:
ROBOFLOW_API_KEY = ""
SOURCES = [
    # ("your-workspace", "board-defects", 1),
]
from roboflow import Roboflow
downloaded = []
if SOURCES:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    for ws, proj, ver in SOURCES:
        ds = rf.workspace(ws).project(proj).version(ver).download("yolov8", location=f"/content/src/{proj}")
        downloaded.append(ds.location)


## 3. Merge, train, export

This is the same merge logic as the parts notebook, run with `DEFECT_CLASSES`. `yolo11n` is enough here: boards are photographed close up.

In [ ]:
MERGED = Path("/content/defects"); shutil.rmtree(MERGED, ignore_errors=True)
for split in ("train", "valid"):
    (MERGED/split/"images").mkdir(parents=True); (MERGED/split/"labels").mkdir(parents=True)
for loc in downloaded:
    root = Path(loc)
    names = yaml.safe_load(open(root/"data.yaml"))["names"]
    if isinstance(names, dict): names = [names[k] for k in sorted(names)]
    remap = {i: (DEFECT_CLASSES.index(to_canonical(n)) if to_canonical(n) else None) for i, n in enumerate(names)}
    for lbl in root.rglob("labels/*.txt"):
        img = next((p for p in lbl.parent.parent.glob(f"images/{lbl.stem}.*")), None)
        if img is None: continue
        lines = [" ".join([str(remap[int(l.split()[0])])] + l.split()[1:]) for l in open(lbl) if len(l.split()) == 5 and remap.get(int(l.split()[0])) is not None]
        if not lines: continue
        split = "valid" if ("valid" in img.parts or "test" in img.parts) else "train"
        shutil.copy(img, MERGED/split/"images"/f"{root.name}_{img.name}")
        open(MERGED/split/"labels"/f"{root.name}_{img.stem}.txt", "w").write("\n".join(lines) + "\n")
yaml.safe_dump({"path": str(MERGED), "train": "train/images", "val": "valid/images", "names": DEFECT_CLASSES, "nc": len(DEFECT_CLASSES)}, open(MERGED/"data.yaml", "w"))

model = YOLO("yolo11n.pt")
model.train(data=str(MERGED/"data.yaml"), epochs=100, imgsz=640, batch=16, patience=25, project="/content/runs", name="defects")
best = YOLO("/content/runs/defects/weights/best.pt")
best.val()
best.export(format="onnx", imgsz=640, simplify=True)
print("Copy /content/runs/defects/weights/best.onnx → public/models/defects.onnx")
